### Streaming
- Streaming is a powerful technique that allows you to process data in real-time as it is generated or received. It is commonly used in applications such as video streaming, audio streaming, and real-time data processing.
- In streaming, data is processed in small chunks or "streams" rather than waiting for the entire dataset to be available.
- This allows for faster processing and reduced latency, making it ideal for applications that require immediate feedback or real-time analysis.
- Streaming can be implemented using various technologies and protocols, such as HTTP Live Streaming (HLS), Real-Time Messaging Protocol (RTMP), and WebSockets.
### Working of Streaming-
1. **Data Source**: This is the origin of the data being streamed. It could be a live video feed, an audio source, or any other type of data that needs to be processed in real-time.
2. **Data Processing**: As the data is generated or received, it is processed in small chunks. This can involve tasks such as encoding, decoding, filtering, or analyzing the data on-the-fly.
3. **Data Transmission**: The processed data is then transmitted to the end-users or clients. This can be done using various streaming protocols that ensure efficient delivery of the data.
4. **Data Consumption**: The end-users or clients consume the streamed data in real-time. This could involve playing a video, listening to audio, or analyzing the data as it is received.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# GROQ Model Integration
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002C868E9E510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002C868E9EF90>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
# GROQ Model Integration
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct")
model

# model.stream("Write me a 100 words paragraph on AI") # Fetches the generator object BaseChatModel.stream
for chunk in model.stream("Write me a 100 words paragraph on AI"):
    print(chunk.text)


### Batch
- Batching is a collection of independent requests to a model can significantly improve the performance and reduce costs, as the processing can be done in parallel.

In [ ]:
# GROQ Model Integration
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:meta-llama/llama-4-scout-17b-16e-instruct")
model

responses = model.batch([
    "Write me a 100 words paragraph on AI",
    "Write me a 100 words paragraph on Machine Learning",
    "Write me a 100 words paragraph on Deep Learning"
],
    config={"max_concurrency": 5}   # Optional, such as maximum concurrency level, temp etc..
)
for res in responses:
    print(res.text)

Here is a 100-word paragraph on AI:

Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. AI systems can perform tasks such as visual perception, speech recognition, and decision-making, often with greater speed and accuracy than humans. The technology has numerous applications, including virtual assistants, self-driving cars, and healthcare diagnostics. As AI continues to evolve, it has the potential to revolutionize industries and transform the way we live and work. However, concerns about job displacement, bias, and ethics have sparked ongoing debates about the responsible development and deployment of AI systems. Rapid progress is expected.
Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. It involves training algorithms on large datasets to identify patterns and make predictions or decisions. Machine learning mo

### Tools
- Tool calls are a powerful feature in LangChain that allow language models to interact with external tools and APIs.
- This enables the model to perform actions, retrieve information, dB search, webSearch etc.. which executes the tasks beyond just generating text.
- Tool calls can be used for a variety of purposes, such as fetching data from a database, making API requests, or even controlling hardware devices.
- Tools are the pairings of 
    - schema,including the name of the tool, description of the tool, input parameters, argument definitions (JSON) required to use the tool.
    - A function or coroutine to execute the tool, which takes the input parameters and performs the desired action, returning the result back to the language model.

In [9]:
from langchain.tools import tool
@tool
def get_weather(location : str) -> str:
    """Get the weather for a given location"""
    return f"The weather in {location} is sunny with a high of 25 degrees Celsius."

model_with_tools = model.bind_tools([get_weather])


In [17]:
response = model_with_tools.invoke("What is the weather in India?")
print(response.text)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")


Tool called: get_weather
Arguments: {'location': 'India'}


### Tool Execution loop

In [26]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the weather for a given location"""
    return f"The weather in {location} is sunny with a high of 25 degrees Celsius."

model_with_tools = model.bind_tools([get_weather])

# step_1:Model generates the tool call
messages  = [{"role": "user", "content": "Whats the weather in India,Banglore?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

    # step_2: Execute the tool call
    tool_result = get_weather.invoke(tool_call["args"])
    messages.append({
        "role": "tool", 
        "content": tool_result, 
        "tool_call_id": tool_call["id"]  # required!
        })
    
    # step_3: Send the tool result back to the model for final response
    final_response = model_with_tools.invoke(messages)
    print(final_response.text)

Tool called: get_weather
Arguments: {'location': 'Banglore, India'}
The weather in Banglore, India is sunny with a high of 25 degrees Celsius.
